# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import json
import os
import re
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

# --- API key dari file .env (dicari dari root project maupun folder notebook/) ---
for env_path in (Path('.env'), Path('../.env'), Path('../../.env')):
    if env_path.exists():
        load_dotenv(env_path)
        break
# http timeout eksplisit agar request yang menggantung gagal cepat dan bisa di-retry
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'], http_options={'timeout': 60000})

def with_quota_retry(fn, max_retries: int = 6, base_delay: float = 20.0):
    """Panggil fn(); retry untuk error rate-limit (429), timeout jaringan, dan error server (5xx)."""
    permanent_tags = ('NOT_FOUND', 'INVALID_ARGUMENT', 'UNAUTHENTICATED',
                      'PERMISSION_DENIED', 'API_KEY_INVALID')
    for attempt in range(1, max_retries + 1):
        try:
            return fn()
        except Exception as exc:
            message = str(exc)
            lowered = message.lower()
            # Error permanen (model salah, API key salah, dsb.) langsung gagal
            if any(tag in message for tag in permanent_tags) or attempt >= max_retries:
                raise
            if '429' in message:
                match = re.search(r'retry in ([\d.]+)s', message)
                delay = float(match.group(1)) + 2 if match else base_delay * attempt
                print(f'  Kuota API terlampaui (429) - tunggu {delay:.0f}s, coba lagi ({attempt}/{max_retries - 1})...')
            else:
                # timeout jaringan / 5xx (DEADLINE_EXCEEDED, dsb.)
                delay = 15
                print(f'  Gangguan API ({message[:60]}) - tunggu {delay}s, coba lagi ({attempt}/{max_retries - 1})...')
            time.sleep(delay)

# --- Path project: adaptif agar bisa dijalankan dari root project maupun folder notebook/ ---
def find_project_root() -> Path:
    for candidate in (Path('.'), Path('..'), Path('../..')):
        if (candidate / 'data' / 'ecommerce_faq_knowledge_base.csv').exists():
            return candidate.resolve()
    raise FileNotFoundError('data/ecommerce_faq_knowledge_base.csv tidak ditemukan')

ROOT = find_project_root()
KB_PATH = ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv'
INDEX_PATH = ROOT / 'data' / 'faiss_index.index'
METADATA_PATH = ROOT / 'data' / 'faq_metadata.json'

# --- Muat knowledge base (read-only: data asli TIDAK diubah) ---
df = pd.read_csv(KB_PATH)

print('Path knowledge base :', KB_PATH)
print('Total entri FAQ     :', len(df))
print('Kolom tersedia      :', list(df.columns))
print('Jumlah kategori     :', df['category'].nunique())
print('\nDistribusi kategori:')
print(df['category'].value_counts().to_string())
print('\nMissing values per kolom:')
print(df.isnull().sum().to_string())
df.head()

Path knowledge base : D:\Rework Academy\Praktek\AI-Bootcamp\assignment-semantic-search-AhnafRifqi\data\ecommerce_faq_knowledge_base.csv
Total entri FAQ     : 59
Kolom tersedia      : ['faq_id', 'category', 'question', 'answer']
Jumlah kategori     : 8

Distribusi kategori:
category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5

Missing values per kolom:
faq_id      0
category    0
question    0
answer      0


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
# Gabungkan question + answer menjadi SATU unit retrievable per entri FAQ
df['content'] = 'Q: ' + df['question'] + '\nA: ' + df['answer']
documents = df['content'].tolist()

print('Jumlah unit retrievable :', len(documents))
print('Rata-rata panjang teks   :', round(df['content'].str.len().mean(), 1), 'karakter')
print('\nContoh unit retrievable pertama:')
print(documents[0])

Jumlah unit retrievable : 59
Rata-rata panjang teks   : 190.2 karakter

Contoh unit retrievable pertama:
Q: Bagaimana cara melakukan pemesanan di platform ini?
A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
EMBEDDING_MODEL = 'gemini-embedding-001'  # model embedding aktif (text-embedding-004 sudah tidak tersedia)
EMBEDDING_DIM = 3072                     # dimensi default gemini-embedding-001

def get_embedding(text: str) -> np.ndarray:
    """Embed SATU teks (dipakai untuk meng-embed query pencarian)."""
    result = with_quota_retry(lambda: client.models.embed_content(model=EMBEDDING_MODEL, contents=text))
    return np.array(result.embeddings[0].values, dtype='float32')

def get_embeddings_batch(texts: list, batch_size: int = 100) -> np.ndarray:
    """Embed DAFTAR teks per batch (1 request API per batch) untuk knowledge base."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        result = with_quota_retry(
            lambda: client.models.embed_content(model=EMBEDDING_MODEL, contents=batch)
        )
        vectors.extend(np.array(e.values, dtype='float32') for e in result.embeddings)
        print(f'  Batch {start // batch_size + 1}: {len(batch)} teks selesai')
    return np.vstack(vectors)

embeddings = get_embeddings_batch(documents)

# --- Verifikasi dimensi/shape sebelum lanjut ke tahap penyimpanan ---
print('Shape embeddings:', embeddings.shape)
assert embeddings.shape == (len(documents), EMBEDDING_DIM), 'Shape embedding tidak sesuai!'
print(f'Verifikasi OK: {embeddings.shape[0]} vektor x {embeddings.shape[1]} dimensi, tipe {embeddings.dtype}')

  Batch 1: 59 teks selesai
Shape embeddings: (59, 3072)
Verifikasi OK: 59 vektor x 3072 dimensi, tipe float32


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [4]:
# Normalisasi vektor agar L2 distance pada FAISS setara dengan cosine similarity
embeddings = np.ascontiguousarray(embeddings, dtype='float32')
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# --- Verifikasi jumlah vektor tersimpan sesuai jumlah entri knowledge base ---
print('Dimensi vektor              :', dimension)
print('Jumlah vektor tersimpan     :', index.ntotal)
assert index.ntotal == len(documents), 'Jumlah vektor != jumlah entri knowledge base!'
print('Verifikasi OK: jumlah vektor == jumlah entri FAQ =', len(documents))

# --- Simpan index + metadata agar bisa dimuat ulang tanpa mengulang embedding ---
faiss.write_index(index, str(INDEX_PATH))
metadata = df[['faq_id', 'category', 'question', 'answer']].to_dict(orient='records')
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('Index FAISS tersimpan di :', INDEX_PATH)
print('Metadata FAQ tersimpan di:', METADATA_PATH)

Dimensi vektor              : 3072
Jumlah vektor tersimpan     : 59
Verifikasi OK: jumlah vektor == jumlah entri FAQ = 59
Index FAISS tersimpan di : D:\Rework Academy\Praktek\AI-Bootcamp\assignment-semantic-search-AhnafRifqi\data\faiss_index.index
Metadata FAQ tersimpan di: D:\Rework Academy\Praktek\AI-Bootcamp\assignment-semantic-search-AhnafRifqi\data\faq_metadata.json


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query: str, top_k: int = 3) -> list:
    """Ubah query menjadi embedding, lalu cari entri FAQ paling relevan di index FAISS."""
    query_embedding = get_embedding(query).reshape(1, -1)
    faiss.normalize_L2(query_embedding)  # format harus sama dengan vektor di dalam index

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx < 0:  # FAISS mengembalikan -1 bila jumlah index kurang dari top_k
            continue
        item = dict(metadata[idx])
        # faiss.IndexFlatL2 mengembalikan squared L2 distance; makin kecil = makin mirip
        item['distance'] = float(distance)
        # vektor ternormalisasi -> cosine similarity = 1 - (squared L2 / 2)
        item['similarity_score'] = 1.0 - float(distance) / 2.0
        results.append(item)
    return results

In [6]:
# Contoh penggunaan semantic search: tampilkan hasil beserta skor masing-masing
sample_query = 'bisa bayar pake cicilan nggak?'
print('Query:', sample_query)
print('-' * 95)
for rank, r in enumerate(semantic_search(sample_query, top_k=3), start=1):
    print(f"{rank}. [distance={r['distance']:.4f} | similarity={r['similarity_score']:.4f}] "
          f"(faq_id={r['faq_id']}, {r['category']})") 
    print(f"   {r['question']}")

Query: bisa bayar pake cicilan nggak?
-----------------------------------------------------------------------------------------------


1. [distance=0.1914 | similarity=0.9043] (faq_id=20, Pembayaran)
   Apakah saya bisa membayar dengan cicilan?
2. [distance=0.5265 | similarity=0.7368] (faq_id=22, Pembayaran)
   Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
3. [distance=0.5705 | similarity=0.7148] (faq_id=17, Pembayaran)
   Metode pembayaran apa saja yang tersedia?


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [7]:
def build_prompt(query: str, search_results: list) -> str:
    """Susun prompt terstruktur: Persona, Task, Context (hasil retrieval), Format."""
    # --- Context: hasil semantic search disisipkan ke dalam prompt ---
    if search_results:
        context = '\n\n'.join(
            f"[FAQ {i + 1}] (kategori: {r['category']})\n"
            f"Pertanyaan: {r['question']}\n"
            f"Jawaban: {r['answer']}"
            for i, r in enumerate(search_results)
        )
    else:
        context = '(Tidak ada FAQ relevan yang ditemukan di knowledge base)'

    prompt = f"""[PERSONA]
Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan membantu, serta menjawab dalam bahasa Indonesia.

[TASK]
Jawab pertanyaan pengguna pada bagian [PERTANYAAN PENGGUNA] HANYA berdasarkan [CONTEXT] di bawah ini.
Aturan wajib:
- Jangan gunakan pengetahuan umum atau informasi apa pun di luar [CONTEXT].
- Jika informasi yang ditanyakan TIDAK tersedia dalam [CONTEXT], katakan dengan jujur bahwa knowledge base kami belum memiliki informasi tersebut dan sarankan pengguna menghubungi customer service resmi.
- Jika konteks mendekati pertanyaan namun tidak persis, gunakan isi konteks tersebut dan sampaikan dengan bahasa alami.

[CONTEXT]
{context}

[PERTANYAAN PENGGUNA]
{query}

[FORMAT]
Jawaban maksimal 3 kalimat dalam bahasa Indonesia dengan gaya percakapan chat yang natural, tanpa menyebut istilah "FAQ", "context", atau "knowledge base".
"""
    return prompt

In [8]:
# Contoh prompt yang akan dikirim ke LLM: context = hasil semantic search
demo_query = 'bisa bayar pake cicilan nggak?'
demo_results = semantic_search(demo_query, top_k=2)
print(build_prompt(demo_query, demo_results))

[PERSONA]
Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan membantu, serta menjawab dalam bahasa Indonesia.

[TASK]
Jawab pertanyaan pengguna pada bagian [PERTANYAAN PENGGUNA] HANYA berdasarkan [CONTEXT] di bawah ini.
Aturan wajib:
- Jangan gunakan pengetahuan umum atau informasi apa pun di luar [CONTEXT].
- Jika informasi yang ditanyakan TIDAK tersedia dalam [CONTEXT], katakan dengan jujur bahwa knowledge base kami belum memiliki informasi tersebut dan sarankan pengguna menghubungi customer service resmi.
- Jika konteks mendekati pertanyaan namun tidak persis, gunakan isi konteks tersebut dan sampaikan dengan bahasa alami.

[CONTEXT]
[FAQ 1] (kategori: Pembayaran)
Pertanyaan: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

[FAQ 2] (kategori: Pembayaran)
Pertanyaan: Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
Jawaban: Saat

## 7. Mengintegrasikan LLM

In [9]:
GENERATION_MODEL = 'gemini-flash-latest'  # alias model Gemini Flash terbaru yang tersedia di API key ini
                                          # (gemini-2.5-flash / gemini-2.0-flash sudah 404 untuk user baru)

def generate_answer(query: str, top_k: int = 3, search_results: list = None) -> tuple:
    """Alur lengkap: query -> retrieval (semantic search) -> prompt -> jawaban LLM."""
    if search_results is None:  # boleh memakai hasil retrieval yang sudah dihitung sebelumnya
        search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = with_quota_retry(lambda: client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.3,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    ))
    return response.text, search_results

# Demo singkat: satu query -> retrieval -> jawaban LLM
answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print('\nSumber FAQ yang digunakan (top retrieval):')
for s in sources:
    print(f"- [{s['distance']:.4f}] {s['question']}")

Jawaban LLM:
Halo! Pembayaran dengan cicilan bisa dilakukan untuk transaksi menggunakan kartu kredit tertentu. Layanan ini berlaku dengan nominal minimal transaksi yang telah ditentukan oleh pihak bank terkait. Ada hal lain yang bisa saya bantu?

Sumber FAQ yang digunakan (top retrieval):
- [0.1783] Apakah saya bisa membayar dengan cicilan?
- [0.5316] Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- [0.5712] Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [10]:
test_queries = [
    # Kategori A — query yang JELAS ada jawabannya di knowledge base (2 query)
    'Bagaimana cara melacak paket yang sedang dikirim?',
    'Berapa lama proses refund setelah pengembalian disetujui?',
    # Kategori B — parafrase/sinonim: kata-kata BERBEDA dari FAQ asli (3 query)
    'Uang saya sudah kepotong tapi kok status pesanan masih belum terbayar, kenapa ya?',
    'Barangku belum nyampe juga padahal udah lewat seminggu, harus saya lakukan apa?',
    'Boleh nggak bayarnya dicicil per bulan?',
    # Kategori C — TIDAK ada jawabannya di knowledge base (1 query)
    'Apakah toko ini menerima pembayaran pakai mata uang kripto seperti Bitcoin?',
]

test_results = []
for q in test_queries:
    print('=' * 100)
    print('QUERY:', q)

    results = semantic_search(q, top_k=3)
    print('--- Hasil retrieval (top 3) ---')
    for rank, r in enumerate(results, start=1):
        print(f"{rank}. [distance={r['distance']:.4f} | similarity={r['similarity_score']:.4f}] "
              f"(faq_id={r['faq_id']}, {r['category']}) {r['question']}")

    # Pakai hasil retrieval yang sama agar query tidak di-embed dua kali (hemat kuota API)
    answer, _ = generate_answer(q, search_results=results)
    print('--- Jawaban LLM ---')
    print(answer)

    test_results.append({
        'query': q,
        'retrieval': [
            {'faq_id': r['faq_id'], 'question': r['question'],
             'distance': round(r['distance'], 4), 'similarity': round(r['similarity_score'], 4)}
            for r in results
        ],
        'answer': answer,
    })

print(f'\nTotal query diuji: {len(test_results)} (minimal 5 sesuai brief)')

QUERY: Bagaimana cara melacak paket yang sedang dikirim?


--- Hasil retrieval (top 3) ---
1. [distance=0.2237 | similarity=0.8882] (faq_id=10, Pengiriman) Bagaimana cara melacak paket yang sedang dikirim?
2. [distance=0.4197 | similarity=0.7901] (faq_id=52, Pengiriman) Kenapa nomor resi saya belum bisa dilacak di website kurir?
3. [distance=0.4498 | similarity=0.7751] (faq_id=12, Pengiriman) Bagaimana jika paket saya belum sampai melebihi estimasi?


--- Jawaban LLM ---
Halo Kak! Untuk melacak paket yang sedang dikirim, Kakak bisa melihat nomor resi yang muncul di halaman 'Pesanan Saya' setelah barang dikirim. Nomor resi tersebut nantinya bisa Kakak gunakan untuk mengecek posisi paket langsung di website kurir terkait ya.
QUERY: Berapa lama proses refund setelah pengembalian disetujui?


--- Hasil retrieval (top 3) ---
1. [distance=0.1694 | similarity=0.9153] (faq_id=25, Pengembalian) Berapa lama proses refund setelah pengembalian disetujui?
2. [distance=0.4544 | similarity=0.7728] (faq_id=27, Pengembalian) Ke mana dana refund akan dikembalikan?
3. [distance=0.5311 | similarity=0.7345] (faq_id=41, Promo) Apakah cashback dari promo langsung masuk ke saldo?


--- Jawaban LLM ---
Halo! Untuk proses pengembalian dana (refund), biasanya membutuhkan waktu 3-7 hari kerja setelah barang yang kamu kembalikan diterima dan diverifikasi oleh penjual. Semoga informasinya membantu, ya!
QUERY: Uang saya sudah kepotong tapi kok status pesanan masih belum terbayar, kenapa ya?


...

--- Jawaban LLM ---
Halo, mohon maaf atas ketidaknyamanannya, ya. Karena paketmu sudah melebihi estimasi lebih dari 3 hari, kamu bisa langsung menghubungi tim customer service kami. Jangan lupa untuk menyertakan nomor resi pengirimanmu agar paket tersebut bisa segera ditelusuri lebih lanjut.
QUERY: Boleh nggak bayarnya dicicil per bulan?


--- Hasil retrieval (top 3) ---
1. [distance=0.2909 | similarity=0.8545] (faq_id=20, Pembayaran) Apakah saya bisa membayar dengan cicilan?
2. [distance=0.5730 | similarity=0.7135] (faq_id=22, Pembayaran) Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
3. [distance=0.5960 | similarity=0.7020] (faq_id=17, Pembayaran) Metode pembayaran apa saja yang tersedia?


--- Jawaban LLM ---
Halo! Pembayaran dengan cicilan bisa dilakukan untuk transaksi menggunakan kartu kredit tertentu, ya. Namun, pastikan transaksi Anda telah memenuhi nominal minimal yang ditentukan oleh pihak bank terkait.
QUERY: Apakah toko ini menerima pembayaran pakai mata uang kripto seperti Bitcoin?


--- Hasil retrieval (top 3) ---
1. [distance=0.5883 | similarity=0.7059] (faq_id=17, Pembayaran) Metode pembayaran apa saja yang tersedia?
2. [distance=0.6514 | similarity=0.6743] (faq_id=22, Pembayaran) Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
3. [distance=0.6819 | similarity=0.6590] (faq_id=19, Pembayaran) Berapa lama batas waktu pembayaran sebelum pesanan dibatalkan?


--- Jawaban LLM ---
Halo Kak! Saat ini metode pembayaran yang kami dukung adalah transfer bank, kartu kredit/debit, e-wallet, dan minimarket rekanan. Informasi mengenai pembayaran dengan mata uang kripto belum tersedia pada sistem kami. Untuk informasi lebih lanjut, Kakak dapat menghubungi tim customer service resmi kami ya!

Total query diuji: 6 (minimal 5 sesuai brief)


### Catatan Analisis Relevansi

Diuji **6 query** (2 eksplisit, 3 parafrase, 1 di luar knowledge base). Detail lengkap: `documentation/retrieval_test_results.md`.

**Temuan utama:**

1. **Retrieval akurat pada semua query yang punya jawaban di KB (5/5 top-1 benar).** Query eksplisit mendapat similarity 0,89–0,92 (faq 10 & 25); ketiga query parafrase juga menemukan FAQ yang tepat di peringkat-1 (0,85–0,91): "uang sudah kepotong" → faq 21, "belum nyampe lewat seminggu" → faq 12, "dicicil per bulan" → faq 20. Ini membuktikan semantic search menangkap **makna, bukan kata kunci** — kata gaul/sinonim yang tidak ada di KB tidak menggagalkan pencocokan.
2. **Query di luar KB menghasilkan skor terendah (0,71 vs 0,85–0,92)** — faq 17 (metode pembayaran) tetap terambil sebagai konteks terdekat, dan LLM **menolak dengan jujur**: menyebut metode yang didukung, menyatakan informasi kripto belum tersedia, lalu menyarankan customer service. Tidak ada halusinasi dari pengetahuan umum — instruksi `[TASK]` pada prompt terpenuhi.
3. **Peringkat 2–3 kadang kurang relevan** (faq 41 cashback untuk query refund; faq 22 split payment untuk query cicilan) — tidak merugikan karena top-1 sudah tepat, tetapi menunjukkan batas `top_k=3` pada knowledge base kecil.
4. **Kualitas jawaban LLM konsisten**: 6/6 sesuai isi FAQ atau sesuai instruksi penolakan, gaya chat natural, maksimal 3 kalimat, tanpa istilah teknis.

**Keterbatasan yang teramati:** belum ada ambang similarity otomatis (sistem selalu mengirim top-k berapa pun skornya; kejujuran penolakan bergantung pada kepatuhan LLM pada prompt), evaluasi hanya 6 query subjektif (bukan benchmark kuantitatif), dan knowledge base hanya 59 entri FAQ pendek tanpa chunking (sesuai lingkup sesi ini).